# Malaysia Tax Laws — Vector Ingestion Pipeline

Loads PDFs from `data/`, enriches them with structured metadata, splits into chunks, and ingests into Qdrant.

**Run cells in order.** Ingestion is idempotent — files already in the store are skipped based on their content hash.

## 1. Imports & Configuration

In [1]:
import os
import sys
import uuid
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
load_dotenv(Path("../.env"))

# Ensure repo root on path for lib.model_provider
sys.path.insert(0, str(Path("../").resolve()))

from lib.model_provider import ModelProviderEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    Filter,
    FieldCondition,
    MatchValue,
    PayloadSchemaType,
    SparseVectorParams,
    SparseIndexParams,
    PointStruct,
    SparseVector,
)

from ingest_utils import (
    DATA_PATH,
    DOCUMENT_REGISTRY,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    get_file_hash,
    get_splitter,
    load_and_chunk_file,
    get_sparse_embedder,
)

COLLECTION_NAME = "malaysia-tax-laws-v2"
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
BATCH_SIZE = 50
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSIONS = 1536

## 2. Document Registry

Each document is explicitly registered with structured metadata. This drives:
- **Filtering** at query time — e.g. search only Public Rulings, or only documents from 2020+
- **Source attribution** in the UI — title and reference number shown alongside answers
- **Authority ranking** — Act (1) > Public Ruling (2) > Guidelines/Announcements (3)
- **Deduplication** — `source_hash` prevents re-ingesting unchanged files

Add new documents here before running ingestion.

In [2]:
# Registry lives in ingest_utils (shared with generate_testset)
print(f"Registry contains {len(DOCUMENT_REGISTRY)} documents")

Registry contains 7 documents


## 3. Client & Splitter Setup

Creates the Qdrant collection if it doesn't exist. **Cosine similarity** is the right distance metric for semantic search — it measures the angle between vectors rather than magnitude, which is what you want when comparing meaning.

In [3]:
embeddings = ModelProviderEmbeddings(model=EMBEDDING_MODEL)
sparse_embedder = get_sparse_embedder()
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

# Delete if exists (clean slate for this collection)
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=EMBEDDING_DIMENSIONS, distance=Distance.COSINE)},
    sparse_vectors_config={
        "sparse": SparseVectorParams(index=SparseIndexParams(on_disk=False))
    },
)
print(f"Created hybrid collection: {COLLECTION_NAME}")

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="metadata.source_hash",
    field_schema=PayloadSchemaType.KEYWORD,
)
print("Payload index ready: metadata.source_hash")
print("Setup complete.")

Deleted existing collection: malaysia-tax-laws-v2


Created hybrid collection: malaysia-tax-laws-v2


Payload index ready: metadata.source_hash
Setup complete.


## 4. Helper Functions

In [4]:
def is_already_ingested(client: QdrantClient, collection_name: str, file_hash: str) -> bool:
    """Returns True if any chunk with this source_hash already exists in the collection."""
    results, _ = client.scroll(
        collection_name=collection_name,
        scroll_filter=Filter(
            must=[FieldCondition(key="metadata.source_hash", match=MatchValue(value=file_hash))]
        ),
        limit=1,
    )
    return len(results) > 0

## 5. Ingest Documents

For each PDF in `data/`:
1. Checks the registry — unregistered files are skipped with a warning
2. Computes a content hash and checks if it's already in the store — skips if so
3. Loads pages, merges registry metadata onto every chunk
4. Ingests in batches of `BATCH_SIZE` to avoid timeouts

In [5]:
pdf_files = sorted(DATA_PATH.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

splitter = get_splitter()
all_chunks = []

for pdf_path in pdf_files:
    registry_entry = DOCUMENT_REGISTRY.get(pdf_path.name)
    if not registry_entry:
        print(f"  WARN  {pdf_path.name} — not in registry, skipping")
        continue

    file_hash = get_file_hash(pdf_path)
    chunks = load_and_chunk_file(pdf_path, registry_entry, splitter, source_hash=file_hash)
    ingested_at = datetime.now(timezone.utc).isoformat()
    for c in chunks:
        c.metadata["ingested_at"] = ingested_at
    all_chunks.extend(chunks)
    print(f"  LOAD  {pdf_path.name} ({len(chunks)} chunks)")

print(f"\nTotal chunks to ingest: {len(all_chunks)}")

# Batch ingest with both dense and sparse vectors
total_batches = (len(all_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]
    batch_texts = [c.page_content for c in batch]

    # Dense embeddings (OpenAI)
    dense_vecs = embeddings.embed_documents(batch_texts)

    # Sparse embeddings (BM42) — batch via embed_documents()
    sparse_results = sparse_embedder.embed_documents(batch_texts)
    # sparse_results is list[tuple[list[int], list[float]]]

    points = []
    for chunk, dense_vec, (sparse_indices, sparse_values) in zip(batch, dense_vecs, sparse_results):
        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector={
                "dense": dense_vec,
                "sparse": SparseVector(
                    indices=sparse_indices,
                    values=sparse_values,
                ),
            },
            payload={
                "page_content": chunk.page_content,
                "metadata": chunk.metadata,
            },
        ))

    client.upsert(collection_name=COLLECTION_NAME, points=points, timeout=60)
    print(f"  Batch {i // BATCH_SIZE + 1}/{total_batches} ingested")

print(f"\nDone. {len(all_chunks)} chunks added to '{COLLECTION_NAME}'.")

Found 7 PDF files



  LOAD  Abroad_income_20240620-guidelines-tax-treatment-in-relation-to-income-received-from-abroad-amendment-june-2024.pdf (65 chunks)


  LOAD  Residence_Status_PR_11_2017.pdf (51 chunks)


  LOAD  Residence_tax_pr-5_2022.pdf (94 chunks)


  LOAD  Tax_releif_PR2_2012.pdf (41 chunks)
  LOAD  Tax_treatment_PR8_2011.pdf (36 chunks)


...

  Batch 31/35 ingested


  Batch 32/35 ingested


  Batch 33/35 ingested


  Batch 34/35 ingested


  Batch 35/35 ingested

Done. 1749 chunks added to 'malaysia-tax-laws-v2'.


## 6. Smoke Test: Verify Hybrid Retrieval

Runs a hybrid RRF query against the new collection to confirm both dense and sparse vectors are indexed and returning results.

In [6]:
# Smoke test: verify hybrid retrieval works on the new collection
from qdrant_client.models import Prefetch, FusionQuery, Fusion, SparseVector as QdrantSparseVector

test_query = "Do the 183 day rule apply if I work remotely?"

# Dense
dense_test = embeddings.embed_query(test_query)
# Sparse
sparse_idx, sparse_vals = sparse_embedder.embed_query(test_query)

results = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=dense_test, using="dense", limit=10),
        Prefetch(
            query=QdrantSparseVector(indices=sparse_idx, values=sparse_vals),
            using="sparse",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
    with_payload=True,
)

print(f"Smoke test: {len(results.points)} results returned")
for p in results.points:
    meta = (p.payload or {}).get("metadata", {})
    print(f"  score={p.score:.4f}  ref={meta.get('reference', '?')}  title={meta.get('title', '?')[:50]}")

assert len(results.points) > 0, "No results returned — ingestion may have failed"
print("\nSmoke test PASSED")

Smoke test: 5 results returned
  score=0.6429  ref=PR 2/2012  title=Tax Relief for Resident Individual
  score=0.5833  ref=PR 2/2012  title=Tax Relief for Resident Individual
  score=0.5000  ref=PR 8/2011  title=Tax Treatment of Employee Benefits
  score=0.4333  ref=PR 2/2012  title=Tax Relief for Resident Individual
  score=0.3111  ref=PR 2/2012  title=Tax Relief for Resident Individual

Smoke test PASSED
